# Train-Time Scaling: Bootstrapping, GRPO, and Stabilizing Long-Chain RL
*How STaR bootstraps reasoning from a model's own correct attempts, DeepSeekMath's GRPO makes RL memory-efficient, and DAPO fixes the implementation details needed to stabilize RL on genuinely hard reasoning problems*

# From Test-Time Scaling to Train-Time Scaling
So far, the ground covered has been test-time scaling (getting more out of a model at inference — repeated sampling, majority voting), tool use and code feedback, and verification (checking which outputs are actually correct). This notebook is about closing the loop: taking what's learned at test time and actually feeding it back into training the model further — **train-time scaling**.

## A Motivating Number
**AIME** is a benchmark of genuinely difficult competition-level math problems — harder than earlier math benchmarks, which had become "solved" (saturated) and were often already present in training data for many models, making them less useful as a fair test.

Some rough numbers on AIME:
- **GPT-3.5** (175B parameters): around **5%** accuracy.
- **DeepSeekMath**, using train-time scaling on a much smaller **7B** model: **51.7%** accuracy — and with additional techniques, up to **60%**.
- A method called **DAPO**, applied to a **Qwen-32B** model: strong accuracy as well.

The surprising part: earlier material emphasized that scaling up parameter count drives capability. But here, a **much smaller** model reaches very high accuracy on a hard reasoning benchmark — not by getting bigger, but through **train-time scaling** techniques. That's the core question this notebook explores: how did these smaller models get so good?


# Why Reinforcement Learning on Reasoning Is Hard to Get Right
Plainly applying reinforcement learning to reasoning tasks often doesn't work well — and the reason is usually **implementation details**, not the underlying idea being flawed. Three key insights guide this notebook:

1. **Models can learn from their own outputs.** Instead of only training on internet text (pre-training), a model can be trained further on carefully filtered versions of its *own* generated outputs — improving itself using its own attempts, when those attempts can be verified as good.
2. **Compute spent training on a model's own outputs can substitute for raw parameter count.** This echoes a pattern seen elsewhere: increasing training data/compute for a smaller model can rival what a much larger model achieves.
3. **Implementation details matter a lot in RL, more so than in plain supervised learning.** Small fixes in how the RL algorithm is set up can make an outsized difference once you try to scale it up.


# Three Training Paradigms
- **Pre-training:** train the model on broad internet-scale text.
- **Fine-tuning:** further training on curated data — this is roughly how chatbots get built, e.g., via RLHF or RLAIF using human (or AI) preference data.
- **Test-time scaling:** inference-only techniques — majority voting, repeated sampling, combining multiple outputs — without changing the model's weights at all.

**Train-time scaling** is what happens when you take the *output* of test-time scaling (after filtering it for quality), and use that filtered output to fine-tune the model further. In short: **generate multiple outputs → filter for the good ones → fine-tune on those → repeat.** That loop is the essence of train-time scaling.


# Evidence: Both Test-Time and Train-Time Compute Help
Looking at reasoning models like o1 on the AIME benchmark: plotting pass@1 accuracy against compute, on a log-linear scale, shows that **more test-time compute** improves accuracy — this was already established. But the same models also showed that **more train-time compute** independently improves pass@1 accuracy as well.

This means there's a genuine loop available: generate model outputs, use them (once filtered/verified) to improve training, and repeat — improving the model on a specific benchmark over successive rounds.

## Why Does This Work Particularly Well in Math?
Math is a domain with strong **verifiability** — it's possible to check with confidence whether an output is actually correct. This means the "filter for good outputs" step of train-time scaling has a reliable way to decide what counts as good. Train-time scaling tends to work better specifically in domains where this kind of verification is possible.


# Why Reasoning (Chain-of-Thought) Helps in the First Place
Reasoning models (o1, Gemini Flash Thinking, and similar) dedicate extra tokens to spelling out a step-by-step process before producing a final answer, rather than jumping straight to an answer. Within that reasoning process, some recurring patterns show up:

- **Problem analysis** — working out what's actually being asked before attempting a solution.
- **Task decomposition** — breaking a problem into smaller sub-tasks.
- **Self-evaluation and backtracking** — checking an intermediate result, recognizing it looks wrong, and going back to redo an earlier step.
- **Trying multiple approaches** — a form of parallel search, trying more than one strategy rather than committing to just one from the start.

## Worked Examples
- **Matrix transpose script:** given a request to write a script that prints the transpose of a matrix, the model first works out the input/output format expected (problem analysis), then explicitly lays out a plan — parse the input, build the matrix, transpose it, print it in the right format (task decomposition). For a simple problem like this, that might be overkill, but for harder problems, this kind of explicit breakdown becomes genuinely useful.
- **Chemistry pH calculation:** partway through computing a pH value, the model catches itself — realizing it was about to use the wrong formula — and corrects course mid-solution. This is self-correction in action, drawing on knowledge the model already had, but catching its own mistake before finalizing an answer.

## Where This Helps Most
Comparing a reasoning ("thinking") model against a non-reasoning model like GPT-4o (measured as win rate — how often humans prefer the reasoning model's output): domains with strong verifiability — programming, data analysis, mathematical calculation — show a win rate clearly above 50%, meaning a real, consistent advantage. Domains like personal writing or text editing show far less improvement, since there's no similarly reliable way to verify which output is "more correct."


# Open Questions About Train-Time vs. Test-Time Scaling

## What's the Right Balance Between Investing in Training-Time vs. Test-Time Compute?
This is a central question this notebook works toward, with a fuller answer building up across all three papers covered here.

## Test-Time Compute Is Cheap and Can Scale Almost Indefinitely — So Why Bother With Train-Time Scaling At All?
Test-time compute is attractive because it's "cheap" in the sense that a model is already trained — you can just run many inferences and, given a good verifier, eventually land on a correct answer through repeated sampling (in principle, this search could even be scaled toward being effectively unlimited). But this only works well in domains where verification is genuinely robust. Where verification is weak or unavailable, simply sampling more doesn't reliably help — you actually need the model itself to reason better, which is where train-time scaling comes in. Both approaches serve a purpose, and the trade-offs between them are explored further using the papers ahead.

**One useful framing:** a pre-trained model already has some baseline capability at solving problems in a given domain. Test-time scaling is like generating many attempts and *picking* the best one using a verifier — effective, but reliant entirely on that verifier being good. Train-time scaling instead tries to directly raise the model's **pass@1** accuracy — making a single attempt more likely to be correct in the first place — though this still benefits from having some verification loop involved in creating good training signal.

## Does Fine-Tuning for Harder Problems Hurt Performance on Easier Ones?
Typically, there shouldn't be meaningful regression on easier problems from training on harder ones — unless the reasoning chains being trained on are flawed in some way, such as repetitive, unproductive reasoning loops (sometimes called "overthinking").


# STaR: Bootstrapping Reasoning With Reasoning

**Paper:** [arxiv.org/abs/2203.14465](https://arxiv.org/abs/2203.14465)

The goal: teach a model to produce good step-by-step reasoning chains, without needing either a massive hand-labeled dataset of reasoning steps, or accepting the weaker performance of just using a few examples via prompting.

## Why This Is Hard to Do With Existing Approaches
- **Internet-scale text rarely contains step-by-step reasoning.** Most written text just states conclusions, not the reasoning process behind them.
- **Manually annotating reasoning steps is expensive** — it requires humans to write out detailed rationales for large numbers of problems.
- **Automating rationale generation from known solution patterns** only works in narrow, well-structured domains.
- **Few-shot prompting** (showing the model a handful of question→rationale→answer examples) underperforms models that are actually fine-tuned on a large dataset — a small number of reasoning examples just isn't enough on its own.


# The Core Idea
Start with a **small** number of example rationales. Use those examples to few-shot prompt the model into generating rationales for a **large** set of problems (say, 10,000). Keep only the rationales that led to a **correct final answer**, and fine-tune the model on those. Then repeat this loop using the improved model.

## The Problem With Just Doing That
If you only ever keep and train on the problems the model already solved correctly, the model never gets any training signal from the problems it **couldn't** solve — it just keeps reinforcing what it could already do, and stops making progress on harder problems.

## The Fix: Rationalization
For problems the model got wrong, give it the **correct answer as a hint**, and ask it to explain, working backward, how one would arrive at that answer. Then fine-tune on this generated rationale too — but without showing the hint at training time, so it looks to the model like it solved the problem directly. This lets the training set expand to include harder problems the model couldn't originally solve on its own, rather than being permanently stuck on only the easy ones.

**In short:** learn from genuinely correct attempts where they exist, and for failures, backfill a plausible-looking reasoning path by working backward from the known answer.


# Key Assumptions Behind STaR
STaR's simplicity rests on a few assumptions:

1. **A correct final answer is a good enough proxy for reasoning quality.** In math specifically, this tended to hold up reasonably well — but it does mean STaR might occasionally train on a *correct answer reached through flawed reasoning*, since there's no separate check on the reasoning steps themselves, only the final answer.
2. **The model can generate a valid reasoning path when given the answer as a hint.** If you show a model the correct answer and ask it to "show its work," it's assumed to be capable of producing a sensible rationale that leads there.
3. **The starting model needs to already be reasonably capable.** If a class of problems is far outside what the initial model can do at all, STaR's iterative bootstrapping won't make much progress — the iterative process helps the model improve *within* its existing capability range, not leap far beyond it.

STaR is sometimes described as a bare-bones, off-policy reinforcement learning technique — it's much simpler than a full RL setup, but functions similarly: generate attempts, keep the good ones, learn from them, repeat.


# How the Algorithm Actually Works
1. Start with a **small** rationale dataset (questions + example rationales + answers), plus a much **larger** training dataset that only has questions and correct answers, without rationales.
2. Few-shot prompt the model, using the small rationale dataset as examples, to generate rationales and answers for questions from the larger training set.
3. Keep only the generated rationales that led to the **correct answer** — this becomes new training data.
4. For questions the model got wrong, use **rationalization**: give the model the correct answer as a hint, have it generate a rationale working backward from that hint, and add this to the training data too (without the hint visible at training time).
5. **Fine-tune** the model on this combined set of rationales.
6. **Repeat** the whole process using the newly fine-tuned model — which should now be able to solve at least some of the previously-unsolved problems, expanding the training set further each round.


# Experimental Setup
STaR was tested using **GPT-J**, a 6-billion-parameter open-source model. Training used a slow warm-up, then a constant learning rate, across a number of outer-loop iterations, gradually increasing the number of steps in the inner loop over time.

## Datasets Used
- **GSM8K** — grade-school math word problems (about 9,000 examples).
- **CommonsenseQA** — everyday multiple-choice reasoning questions.
- **Arithmetic problems** — synthetic multi-digit addition problems generated by the authors themselves.


# Results

## On Arithmetic and CommonsenseQA
STaR boosted performance meaningfully compared to plain supervised fine-tuning, while using **less data** than a fully supervised setup would require — for example, one comparison showed STaR reaching a target accuracy level using only around 70–87% of the available data, having discarded the rest for lacking a correct final answer.

## On CommonsenseQA Specifically
Human raters were shown STaR-generated rationales versus rationales from the original small example set, and asked which they preferred. STaR's rationales generally held up well — the qualitative reasoning came across as reasonable, likely because CommonsenseQA involves everyday, natural-language reasoning rather than precise multi-step math.

## On GSM8K (Math), the Story Was Different
Rationalization didn't meaningfully help on GSM8K — STaR did outperform baseline approaches overall, but the extra rationalization step specifically added little benefit. One likely explanation: GSM8K's problems were largely within GPT-J's existing capability range, so simply forcing chain-of-thought reasoning already got most of the available benefit — there wasn't much extra room for the hint-based rationalization step to add on top of that.

## A Practical Limitation
Running this loop for many iterations tends to **plateau** — progress slows and eventually stalls, meaning careful attention has to be paid to how many iterations are actually worth running, and this isn't quite the same as a fully-fledged RL setup with a proper close feedback loop.


# Takeaways and Open Challenges
- Rationalization only really helps in the specific case where the model is likely capable of producing a good explanation once it already knows the answer — cases where the model was already fairly confident don't benefit as much.
- **Few-shot prompt style matters and introduces bias.** Since the model learns to generate rationales in the style of the few examples it was shown, the specific way those example rationales were written can noticeably shape what kind of reasoning the model ends up producing and training on — a form of prompt-engineering bias baked into the whole pipeline.
- The overall approach generalizes across problem types — symbolic, natural language, and mathematical reasoning were all tested.
- **A core open challenge:** there's no strong way to directly evaluate rationale *quality* for most tasks — correctness of the final answer is used as a stand-in, but this can let through correct answers reached via genuinely flawed reasoning (false positives), since there's no separate check on the reasoning steps themselves.


# Open Questions About STaR

## What Actually Bounds STaR's Performance?
STaR can't make logical leaps that go meaningfully beyond what's implicit in its training data — the quality of rationalization is fundamentally bounded by the model's existing reasoning ability. If a model genuinely can't reason its way to a hinted answer, rationalization won't produce a meaningful rationale for that case either, and no new capability gets introduced through this process. Some answers are also inherently easier to rationalize backward from than others (e.g., a final answer like 225 might hint fairly directly at a multiplication of 15×15, while other numbers offer far less structural hint).

## What About Learning From Negative (Failed) Examples?
STaR only really has a clean way to learn from successes — problems solved directly, or rationalized after the fact once the answer is known. Learning meaningfully from cases that stayed wrong even with hints is a much less solved problem — some later work has attempted this, but it's a harder, still-open direction, since without some non-zero reward signal, there isn't yet a clean way to close that part of the loop.

## Is There Filtering on the Rationalized (Hinted) Examples?
The original paper doesn't filter rationalization outputs beyond checking the final answer — meaning it's possible to end up training on a rationale that reaches the correct hinted answer via flawed intermediate reasoning. Follow-up work has explored using a process reward model on top of these reasoning chains to catch this — but the original STaR paper doesn't include that extra layer.

## Why Not Just Use a Bigger/Frontier Model to Generate This Data Instead?
If the goal is simply to get higher-quality training data for a smaller model, using a larger, more capable model to generate that data (i.e., distillation) will generally produce higher-quality results in practice. STaR's value here isn't about being the most practically efficient way to improve a small model today — it's about showing what's possible when a model bootstraps its own reasoning ability from scratch, without relying on a stronger external model.


# Related Follow-On Work
- **V-STaR** — adds an explicit **verifier** into the loop alongside the generator, rather than relying purely on final-answer correctness as the only signal.
- **Quiet-STaR** — instead of generating reasoning steps as explicit language ("in English"), moves the reasoning process into the model's internal, latent space (via MLPs) — letting the model "think" internally rather than needing to spell out every reasoning step in natural language.


# DeepSeekMath: Getting Train-Time RL Right

**Paper:** [arxiv.org/abs/2402.03300](https://arxiv.org/abs/2402.03300)

On the (older, easier-than-AIME) MATH benchmark, accuracy had generally tracked model size — bigger models did better. Then DeepSeekMath's **7B** model suddenly did far better than expected for its size. The main breakthrough wasn't a bigger model — it was **getting the reinforcement learning part of train-time scaling right**.

## First Ingredient: Better Data, From a Better Starting Point
Earlier work that tried to boost math performance (built on top of the PaLM model) focused on training on large amounts of arXiv papers and other STEM data. DeepSeekMath took a different approach:
- Instead of starting from a general base model, they started from **DeepSeek-Coder** — a model already trained to be good at code. A model with strong code/reasoning ability going in turned out to make later math training more effective.
- Instead of relying mainly on arXiv papers (which, surprisingly, turned out not to have particularly strong or broad math content), they carefully mined and curated math content from **Common Crawl web pages** — producing much larger and higher-coverage math training data than the arXiv-focused approach.

This connects to a broader theme: if a model isn't already reasonably strong in a target domain, the domain-specific capability often needs to be built up first (through supervised fine-tuning on curated, relevant data) — before RL training on top of it can work well.


# Second Ingredient: GRPO — A Cheaper Way to Do RL
The standard RL algorithm used for RLHF-style training (even with verifiers) is **PPO**. PPO's problem at scale: it requires keeping around several full model copies simultaneously — an old policy, a new policy, a separate critic model, and a reward model. This is fine for smaller models, but becomes a serious memory bottleneck as you try to scale RL up to larger models.

## The GRPO Fix
DeepSeekMath introduced **GRPO (Group Relative Policy Optimization)**, which removes the need for a separate critic model entirely — cutting the number of model copies needed from four down to roughly three (or fewer).

**How it works:**
1. For each question, sample a **group** of answers (rather than just one).
2. Score each answer using a reward model.
3. Normalize each answer's reward relative to the group: **(reward − mean of group rewards) ÷ standard deviation of group rewards.**
4. This normalized value becomes the **advantage** used to update the model — essentially, comparing each answer to how well the rest of the group did, rather than relying on a separately-trained critic to estimate a baseline value.

**Why this makes sense:** reward models are usually trained on *comparisons* between responses in the first place, so scoring a group of answers relative to each other fits naturally with how reward models already work — and it removes an entire extra model (the critic) from the training loop, saving significant memory and making RL practical to scale up further.

## Results
Using GRPO, DeepSeekMath's accuracy on the MATH benchmark went from **46.8% to 51.7%** — reportedly the first open-source model at the 7B scale to cross 50% on this benchmark, achieved **without needing a separate critic model** at all.


# How GRPO Compares to Earlier Approaches
Different RL-style variants for improving reasoning mostly differ along two dimensions: **where the training data comes from**, and **how the gradient (update signal) is computed**.

| Approach | How reward/signal works |
|---|---|
| **STaR** | Generate once; reward is 1 if correct, 0 if wrong (binary, one-shot) |
| **Online rejection fine-tuning** | Generate online (using the current model), still binary reward — keep correct attempts, reject incorrect ones |
| **GRPO** | Generate multiple samples per question online, score each with a reward model, and compute a real-valued **advantage** by comparing each sample's reward against the group's mean and spread — a richer, more graded signal than a simple 0/1 |

GRPO's group-based advantage estimation is well suited to single-step problems (like getting a math answer right or wrong), since the "group" naturally provides the comparison baseline that a separate critic model would otherwise need to estimate.


# Open Questions About GRPO

## Doesn't Fine-Tuning on Hard Problems Risk Hurting Performance on Easy Ones?
This connects to a broader concern: if you're updating all the model's weights to get better at hard problems, could that cause regression on simpler problems it already handled well?

The key mechanical issue GRPO's normalization runs into: **if all the sampled answers to a given problem get the same reward (all 0s, or all 1s), there's no useful learning signal** — the normalization (subtracting the mean, dividing by the spread) simply breaks down when there's no variation to work with. This means the training set needs to include problems with a genuine **mix** of difficulty — not so easy that the model always gets a reward of 1, and not so hard that it always gets a reward of 0 — otherwise there's nothing to "hill climb" against for that specific problem.

For the specific concern about catastrophic regression on unrelated skills: this is partly addressed by adding a **KL-divergence penalty** to the training objective — a term that discourages the updated model's behavior from drifting too far from its original policy, which helps preserve capabilities the model already had rather than letting it over-optimize narrowly on the new reward signal. It's also increasingly common that not all model weights strictly need to be updated to introduce a new capability — some more recent approaches use more targeted, lower-footprint updates instead of full fine-tuning.

## Where Does the Reward Score Actually Come From?
A trained **reward model** provides the score for each sampled answer — this doesn't have to be strictly binary (0 or 1); it can also be a more graded score that then gets averaged/normalized within the group, giving a richer signal than a simple correct/incorrect judgment alone.


# What Actually Improved: Consistency, Not Raw Capability
One notable finding: online RL training (sampling from the current model during training, as GRPO does) clearly beat the STaR-style approach. But specifically, what improved was **majority-at-k accuracy** (if you sample many times — say, 32 — how often does the *majority* of those samples land on the correct answer) — **not** raw **pass@k** (whether *at least one* of many samples was correct).

In other words: the model didn't fundamentally become "smarter" in the sense of being able to solve problems it previously couldn't solve at all in any of its samples. It became **more consistent** — more of its individual attempts converged on the correct answer, rather than only occasionally stumbling onto it. This distinction matters for correctly interpreting what train-time RL is actually improving.


# DAPO: Fixing GRPO's Rough Edges on Harder Problems

**Paper:** [arxiv.org/abs/2503.14476](https://arxiv.org/abs/2503.14476)

Naively applying GRPO at scale — even on an easily-accessible model like Qwen-32B — only reached about **30%** on the AIME benchmark. Several specific problems showed up: the model's **entropy collapses** (it becomes overly confident and stops exploring different approaches), training becomes **unstable**, and the model's **response length becomes uncontrollable** (exploring in an unbounded, runaway way). By comparison, DeepSeek's own R1 paper reported reaching 47% on AIME. DAPO set out to make explicit exactly which RL implementation details were needed to close this gap and go further — details that weren't fully spelled out in earlier work.

## Problem 1: PPO's Clipping Is Too Symmetric
Standard PPO-style clipping treats increasing and decreasing a token's probability the same way. This means a low-probability token can only be pushed up by a limited amount before being clipped, while high-probability tokens also get clipped in the same symmetric way — and this symmetric clipping causes exploration to collapse over time (the model stops trying different things and gets stuck being overconfident on a narrow set of outputs).

**DAPO's fix — "Clip-Higher":** make the clipping **asymmetric**, allowing bigger increases in probability than decreases. This kept accuracy meaningfully higher and kept entropy (a proxy for how much the model is still exploring) much more stable, instead of collapsing.


# Problem 2: Wasted Gradient From Uninformative Groups
Recall from GRPO: for each question, sample a group of answers, and normalize each one's reward relative to the group's mean and spread. But if an entire group of sampled answers is **all correct** or **all wrong**, that normalization has nothing to work with — there's no useful gradient signal to learn from, and that group is essentially wasted compute.

**DAPO's fix — "Dynamic Sampling":** oversample more responses than needed, then explicitly **filter out** any group where all samples got a reward of 1 (all correct) or all got a reward of 0 (all wrong), keeping only the groups that show a genuine mix of outcomes. This keeps the *effective* batch size meaningful — every group that survives the filter actually contributes a usable learning signal, rather than diluting the batch with groups that have nothing to teach the model.


# Problem 3: Long Garbage Answers Get the Same Weight as Short Good Ones
If loss is computed at the **sample level** (one loss value per full question-answer sequence), then a very long, rambling, low-quality answer ends up counted the same as a short, clean, correct one — length differences aren't accounted for at all.

**DAPO's fix — Token-Level Loss:** compute the loss at the **token level** instead of the sample level, so that response length is naturally reflected in the loss rather than every full response being treated as one equal-weight unit regardless of how long or short it is. This helped keep both entropy and the model's average response length under better control, rather than growing unchecked.


# Problem 4: Truncated Reasoning Chains Add Noise
On harder problems, the model sometimes runs out of space mid-thought — its reasoning chain gets cut off (truncated) before it reaches a natural conclusion, simply because it hit a length limit while still "thinking." These truncated, incomplete outputs can introduce noisy, misleading training signal.

**DAPO's fix — Soft Overlong Punishment:** apply a **gradual penalty** based on how far a response runs past the ideal length, rather than a harsh cutoff — helping training stay stable instead of being thrown off by noisy, truncated reasoning chains. This is one of a few different strategies other work has used to handle this same issue; another common approach elsewhere is simply increasing the allowed context length during RL training instead.


# Putting It All Together: Results
Starting from plain GRPO on the AIME benchmark, adding each DAPO technique one at a time on top of the Qwen model showed cumulative gains:

| Configuration | AIME accuracy |
|---|---|
| Plain GRPO (baseline) | 30% |
| + Overlong filtering | 36% |
| + Asymmetric clipping (Clip-Higher) | 38% |
| + Soft overlong punishment | 41% |
| + Token-level loss | 42% |
| + Dynamic sampling | **50%** |

The fully combined DAPO recipe reached **50%** on AIME 2024 using the Qwen2.5-32B base model — reportedly outperforming DeepSeek-R1-Zero-Qwen-32B's reported 47% score, while using around half as many training steps to get there.


# Key Lessons From DAPO
- **The loss function alone isn't a reliable enough signal to monitor during RL training.** You also need to actively track:
  - **Response length** — is it exploding uncontrollably?
  - **Entropy** — is it staying in a healthy range (not too low, which signals collapsed exploration; not too high, which signals the model isn't converging on anything useful)?
  - **The fraction of samples achieving full accuracy** — this tells you how much oversampling/filtering you actually need to maintain a usable batch.
- **If accuracy stalls after a number of steps**, the reward model itself might have become saturated (i.e., it's no longer providing a meaningfully different signal across samples) — the problem may not be the policy, but the reward signal it's being trained against.

**Bottom line on RL vs. supervised fine-tuning:** in domains with a strong, reliable reward signal, RL lets a model improve using relatively few examples — but getting the implementation genuinely right takes real, careful engineering work. Supervised fine-tuning is often faster to get some improvement out of when high-quality labeled data is available, but it doesn't build new reasoning capability in the same way RL-based approaches can.


# Comparing the Three Techniques Covered in This Notebook
| Situation | Best fit |
|---|---|
| Only a small number of example rationales (~100), no RL infrastructure available | **STaR** — a good starting point, especially for simpler reasoning tasks like GSM8K |
| A reasonably strong base model, limited GPU/memory resources | **GRPO (DeepSeekMath)** — a solid, memory-efficient RL algorithm, though it still needs good instruction-tuned data to prime the model first |
| Long reasoning chains, competition-level problems (AIME, IMO-style), need state-of-the-art results | **DAPO** — requires carefully controlling clipping, sampling, response length, and entropy, but unlocks the strongest results on the hardest problems |

## What These Techniques Do and Don't Improve
Across all three approaches, what reliably improves: **majority-at-k accuracy** (as more compute is applied), answer formatting quality, and coherence across multi-step reasoning. What generally does **not** improve: the model's **fundamental capability** to solve genuinely new kinds of problems, or meaningfully generalize far outside its existing domain — none of these train-time scaling techniques teach a model something it couldn't do at all in any of its samples to begin with.


# Open Questions and Active Research Directions

## Why Does Majority-at-K Improve But Not Pass@K?
This remains a genuinely open question. The working intuition: improving pass@k (whether *any* sample is ever correct) would require the model to gain a fundamentally new capability to solve problems it previously couldn't solve in any of its attempts — and fundamental capability jumps have historically come from either a genuine algorithmic breakthrough, or scaling along some dimension (more parameters, more/better data), not from re-weighting existing behavior through this kind of RL fine-tuning alone.

## Are Reasoning Behaviors Like Backtracking and Self-Correction Genuinely New, or Just More Frequent?
An open question: when a model shows behaviors like backtracking or self-evaluation more often after this kind of training, is that a genuinely new capability being introduced, or was that capability already present in the base model and simply becoming statistically more common/prominent through training? This isn't fully settled.

## Learning From Failures Remains Mostly Unsolved
Right now, most of these techniques primarily work by **filtering out** failures (all-wrong groups, incorrect rationales) rather than learning something useful directly from them. A few papers have explored ways to extract useful signal from failed attempts, but this remains a comparatively underdeveloped direction.

## Where Do Verification Signals Come From, and Are They Good Enough?
A recurring theme: math and code are domains where a verifiable, checkable reward is naturally available (a final correct answer, execution feedback, unit tests). Many other real-world domains don't have this kind of clean signal — an important, still-open question is which other domains can be given verification signal, and whether combining multiple, imperfect verifiers (an ensemble, as covered in earlier verification research) can help make up for what any single verifier misses.

## Promising Directions Going Forward
- **Better training data generation** — improving the quality and coverage of what gets fed into these loops.
- **Making RL robust to noisy reward models** — since reward model imperfection is a real, recurring bottleneck across all of these approaches.
- **Combining techniques across papers** — e.g., blending STaR-style rationalization with DAPO-style RL stabilization techniques, rather than treating them as fully separate approaches.
